# Phase 5 — NB2: Stage 2 Sentiment Training (SemEval 2014)

**Goal:** Train Stage 2 Sentiment Classification:
1. **Retrieval (Phase 2a):** `stage2_2014.yaml` — diagonal W, tau=0.3, lambda_rank=0.01
2. **No-Retrieval baseline:** `stage2_2014_noret.yaml`

**Dataset:** SemEval 2014 Task 4 Restaurant (3 polarities: positive, negative, neutral)

**Input:**
- `lcminhc/p5-nb1-stage1` — processed data (from NB1)
- `lcminhc/p5-embed-v4` — embedding ckpt (from NB0)

**Output:** Upload `/kaggle/working/outputs_p5_nb2/` as Kaggle dataset `p5-nb2-stage2`
- `stage2_2014_best.pt` (retrieval)
- `stage2_2014_noret_best.pt` (no-retrieval)
- Training logs

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml

In [ ]:
import os, sys, json, shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
def find_input(name):
    for p in [f'/kaggle/input/{name}', f'/kaggle/input/datasets/lcminhc/{name}']:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'Dataset {name} not found')

NB1 = find_input('p5-nb1-stage1')
EMB = find_input('p5-embed-v4')
print(f'NB1: {NB1} -> {os.listdir(NB1)}')
print(f'EMB: {EMB} -> {os.listdir(EMB)}')

# Copy processed data from NB1
os.makedirs('data/processed', exist_ok=True)
for f in ['sentiment_records.jsonl', 'category_detection.jsonl']:
    shutil.copy(f'{NB1}/{f}', f'data/processed/{f}')
    with open(f'data/processed/{f}') as fp:
        n = sum(1 for _ in fp)
    print(f'data/processed/{f}: {n} records')

# Copy embedding checkpoint from NB0
os.makedirs('checkpoints/embedding_2014', exist_ok=True)
shutil.copy(f'{EMB}/embedding_best.pt', 'checkpoints/embedding_2014/best.pt')
print(f'Embedding ckpt: {os.path.getsize("checkpoints/embedding_2014/best.pt") / 1e6:.1f} MB')

## 0b. Build FAISS Index

Rebuild from `sentiment_records.jsonl` (train split) using embedding model.

In [ ]:
os.makedirs('indexes', exist_ok=True)
!python scripts/03_build_index.py \
    --embedding_ckpt checkpoints/embedding_2014/best.pt \
    --input data/processed/sentiment_records.jsonl \
    --out_dir indexes/

for f in ['train.faiss', 'metadata.json', 'train_vectors.npy']:
    path = f'indexes/{f}'
    if os.path.exists(path):
        print(f'{f}: {os.path.getsize(path)/1e6:.2f} MB')
    else:
        print(f'MISSING: {f}')

In [ ]:
import torch, gc
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Train — Retrieval (Phase 2a)

Config: `stage2_2014.yaml`
- Diagonal W (256 params), tau=0.3, lambda_rank=0.01
- encoder_lr=2e-6, head_lr=2e-4, 20 epochs, patience=7

In [ ]:
gc.collect()
torch.cuda.empty_cache()
!python scripts/04b_train_stage2.py \
    --config configs/stage2_2014.yaml \
    --embedding_ckpt checkpoints/embedding_2014/best.pt \
    --index_dir indexes/ \
    --retrieval_config configs/retrieval_v2.yaml

In [ ]:
# Save retrieval checkpoint immediately
output_dir = '/kaggle/working/outputs_p5_nb2'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f'{output_dir}/logs', exist_ok=True)
src = 'checkpoints/stage2_2014/best.pt'
if os.path.exists(src):
    shutil.copy(src, f'{output_dir}/stage2_2014_best.pt')
    print(f'stage2_2014_best.pt: {os.path.getsize(src)/1e6:.1f} MB')
if os.path.exists('logs/stage2_2014_training.jsonl'):
    shutil.copy('logs/stage2_2014_training.jsonl', f'{output_dir}/logs/')
    print('Retrieval log saved')

## 2. Train — No-Retrieval Baseline

Config: `stage2_2014_noret.yaml`
- No retrieval, 20 epochs, patience=7

In [ ]:
gc.collect()
torch.cuda.empty_cache()
!python scripts/04b_train_stage2.py \
    --config configs/stage2_2014_noret.yaml \
    --no_retrieval

## 3. Training Logs

In [ ]:
for label, log_path in [('Retrieval (Phase 2a)', 'logs/stage2_2014_training.jsonl'),
                        ('No-Retrieval',         'logs/stage2_2014_noret_training.jsonl')]:
    print(f'\n=== {label} ===')
    if not os.path.exists(log_path):
        print('No log found.')
        continue
    print(f'{"Epoch":<6} {"Loss":<8} {"Acc":<8} {"MacF1":<10} {"pos":<7} {"neg":<7} {"neu":<7}')
    print('-' * 55)
    with open(log_path) as f:
        for line in f:
            r = json.loads(line)
            print(f"{r['epoch']:<6} {r['train_loss']:<8.4f} "
                  f"{r['sentiment_acc']:<8.4f} {r['sentiment_macro_f1']:<10.4f}"
                  f"{r.get('f1_positive', 0):<7.3f} "
                  f"{r.get('f1_negative', 0):<7.3f} "
                  f"{r.get('f1_neutral', 0):<7.3f}")

## 4. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p5_nb2'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f'{output_dir}/logs', exist_ok=True)

# Retrieval checkpoint
src = 'checkpoints/stage2_2014/best.pt'
if os.path.exists(src):
    shutil.copy(src, f'{output_dir}/stage2_2014_best.pt')
    print(f'stage2_2014_best.pt: {os.path.getsize(src)/1e6:.1f} MB')

# No-retrieval checkpoint
src = 'checkpoints/stage2_2014_noret/best.pt'
if os.path.exists(src):
    shutil.copy(src, f'{output_dir}/stage2_2014_noret_best.pt')
    print(f'stage2_2014_noret_best.pt: {os.path.getsize(src)/1e6:.1f} MB')

# Logs
for log in ['stage2_2014_training.jsonl', 'stage2_2014_noret_training.jsonl']:
    src = f'logs/{log}'
    if os.path.exists(src):
        shutil.copy(src, f'{output_dir}/logs/{log}')

print(f'\nOutputs saved to {output_dir}')
print('Upload as Kaggle dataset: p5-nb2-stage2')

In [ ]:
shutil.make_archive('/kaggle/working/outputs_p5_nb2_backup', 'zip',
                    '/kaggle/working', 'outputs_p5_nb2')
size_mb = os.path.getsize('/kaggle/working/outputs_p5_nb2_backup.zip') / 1e6
print(f'Backup zip: {size_mb:.1f} MB')